# Step 1: train accuracy vs validation accuracy

Drives `src/trainval.py` over one owner's half of the grid. Every checkpoint is scored on
**its own training set** and compared to the validation accuracy already recorded - **no
retraining**, and no cache rebuild.

One question: is the 0.908 diagonal a variance problem or a bias problem?

| Result | Meaning | What helps |
|---|---|---|
| train ~ 1.00, val ~ 0.91 | **Overfitting** (variance-limited) | More data, more augmentation. **A bigger model makes it worse.** |
| train ~ val ~ 0.91 | **Underfitting** (bias-limited) | More capacity - wider, longer |

`load_arm` is seed-deterministic, so reloading with the same seed reproduces byte-identical
training rows, and scoring reuses `train._predict` - the same eval path as every reported cell.
The recomputed validation accuracy therefore reproduces what training recorded, and the check
reports how many **validation rows** disagree, allowing up to 4 per checkpoint before it raises.

The budget is not zero on purpose: `_predict` runs under fp16 autocast, so a score within
rounding distance of the decision threshold can flip on a different GPU than the one that
trained the run. A wrong arm, seed or cache moves tens of rows instead, and then the train
accuracy beside it describes different images and means nothing.

**This notebook is meant to be run more than once.** If the D4 ablation goes ahead, point
`--results-dir` and `--ckpt-root` at that grid and pass `--tag d4`: the before/after on this
gap is what says whether augmentation actually closed it, and the tag keeps the two files
from overwriting each other. The `TAG` variable in the setup cell is the only thing to change.

## Setup

Set `OWNER` in the next cell - and `TAG`/`RESULTS`/`CKPT_ROOT` only when scoring a second
grid. Same paths as `01_run_matrix.ipynb`: the cache is read from the shared root, the
checkpoints from **this account's own** Drive backup.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob
from pathlib import Path

# A CPU runtime will run this - slowly, and `--device cuda` would fail outright. On a CPU
# runtime nvidia-smi is not installed at all, hence the fallback message.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU on this runtime -> Runtime > Change runtime type > T4 GPU > Save, then re-run this cell (the restart unmounts Drive)"

from google.colab import drive
drive.mount('/content/drive')

# Who is running this notebook. Only this account's checkpoints have weights here; the
# other half is skipped, which is expected rather than an error.
OWNER = "ido"

# Read root, shared between both accounts - identical bytes, nothing copied.
DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# Write root, this account's OWN My Drive. Deliberately a different tree from CACHE so a
# view-only share cannot break it - see 01_run_matrix.ipynb, setup cell.
BACKUP = f"/content/drive/MyDrive/deep_learning_results/{OWNER}"

# The three lines to change when scoring a second grid. TAG suffixes the output file, so a
# D4 pass cannot overwrite the flip-only numbers it is being compared against.
TAG = ""                      # "" for the reported grid, "d4" for the ablation
RESULTS = "results/runs"      # results/runs_d4 for the ablation
CKPT_ROOT = f"{BACKUP}/runs"  # {BACKUP}/runs_d4 for the ablation

assert os.path.isdir(DRIVE), (
    f"not found: {DRIVE}\nCheck it is mounted on THIS account: `ls /content/drive/MyDrive`."
)

print(f"owner   {OWNER}{'  tag ' + TAG if TAG else ''}")
print("cache  <-", CACHE, "" if os.path.isdir(CACHE) else "  <- MISSING")
print("ckpts  <-", CKPT_ROOT)

## Get the code

In [ ]:
# Idempotent: clones on the first run, pulls on every later one.
![ -d /content/deep-learning ] || git clone -q https://github.com/noa-keter/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git pull --ff-only
!git log --oneline -1

## Gate: are this account's checkpoints actually there?

`*.pt` is gitignored and `/content` is wiped on a runtime recycle, so the only surviving
weights are the Drive backup written by `01_run_matrix.ipynb` cell 17. Twenty-eight is a full
half of the reported grid; anything less means the summary quietly averages over fewer runs.

In [ ]:
found = sorted(glob.glob(f"{CKPT_ROOT}/*/*/seed*/model.pt"))
metrics = sorted(glob.glob(f"{RESULTS}/*/*/seed*/metrics.json"))

by_strategy = {}
for path in found:
    by_strategy.setdefault(Path(path).parts[-4], []).append(path)

print(f"{len(metrics)} metrics.json from git")
print(f"{len(found)} checkpoints under {CKPT_ROOT}")
for strategy, paths in sorted(by_strategy.items()):
    print(f"  {strategy:<13}{len(paths):>3}")

if not found:
    print("\nNOTHING FOUND. Is OWNER right, and is Drive mounted on THIS account?")

## Run it

One `python -m src.trainval` subprocess, exactly as the matrix is driven: VRAM is fully
released at the end and a failure leaves the notebook alive to report it.

Scoring the training set is the expensive half - 3,600 images per checkpoint on top of the
400-image val set - but it is inference only, so it is bounded by VRAM rather than by
optimization and runs in seconds per checkpoint.

In [ ]:
import shutil

# `--out-dir` points at Drive rather than the clone. The default is `results/figures` under
# /content, which a runtime recycle wipes - and this run costs 20-30 minutes, so losing it
# to a disconnect means paying that twice. Writing to Drive first makes the result outlive
# the runtime; the copy below puts it where cells 10 and 12 already look, so the merge and
# the `git add` are unchanged. Same split as 05_ensemble.ipynb: Drive holds the artifact,
# git holds what the report cites.
OUT_DIR = f"{BACKUP}/trainval"
os.makedirs(OUT_DIR, exist_ok=True)

tv_cmd = (
    f'python -m src.trainval --cache-dir "{CACHE}" --ckpt-root "{CKPT_ROOT}"'
    f' --owner {OWNER} --results-dir "{RESULTS}" --out-dir "{OUT_DIR}"'
    + (f' --tag {TAG}' if TAG else '')
)
!{tv_cmd}

# `!` does not raise, so without this a failed run would fall through to the merge cell and
# report over whatever files happen to be lying in results/figures.
written = Path(OUT_DIR) / f"train_val_gap_{OWNER}{f'_{TAG}' if TAG else ''}.json"
assert written.is_file(), f"nothing written to {written} - read the error above"

os.makedirs("results/figures", exist_ok=True)
shutil.copy2(written, "results/figures/")

print(f"\nDrive -> {written}  ({written.stat().st_size / 1024:.1f} KB, survives a restart)")
print(f"clone -> results/figures/{written.name}  (this is the copy the last cell commits)")

## Both halves together

Each account commits its own `train_val_gap_<owner>.json`. Once both are on `main`, this cell
merges every file matching the current `TAG` and re-prints the table over all four arms.

Read the **gap** column, not the train column alone: the gap is what caps what more data or
augmentation could buy. In the flip-only grid it was +0.057 for `center_crop` and +0.077 for
`rescale`, against a cross-generator drop of 0.255 and 0.342 - which is why the variance
channel is the smaller of the two and why Gate 0 exists before any ablation is run.

In [ ]:
from src.trainval import format_report, verdict

PREFIX = "train_val_gap_"
rows = []
for path in sorted(glob.glob(f"results/figures/{PREFIX}*.json")):
    # The tag is parsed off rather than globbed for: `*_ido.json` and `*_ido_d4.json` both
    # match any wildcard loose enough to catch two owners, and mixing two grids into one
    # table would be silent nonsense rather than an error.
    owner, _, tag = Path(path).stem[len(PREFIX):].partition("_")
    if tag != TAG:
        continue
    part = json.loads(Path(path).read_text())
    rows += part
    print(f"{Path(path).name:<40}{len(part):>3} checkpoints")

arms = sorted({r["strategy"] for r in rows})
print(f"\n{len(rows)} checkpoints over {len(arms)} arms: {', '.join(arms)}\n")
print(format_report(rows))
print("\nVERDICT:", verdict(rows))

## Hand off

The JSON goes through git, like `metrics.json` - a few KB, and it is what merges the two
halves. A copy also stays in Drive at `<BACKUP>/trainval/`, which is **not** a second source
of truth: git is what the report cites, and the Drive copy exists only so a disconnect during
the 20-30 minute run does not cost the whole run. If the two ever disagree, the git one is
the one that was reviewed.

If the verdict says **variance-limited**, the next step is *not* the ablation. Run Gate 0
first - `notebooks/03_rotation_sensitivity.ipynb`, zero training, minutes - because it decides
for free whether D4 has anything to give at all.

In [ ]:
!git status --short results/figures
suffix = f"_{TAG}" if TAG else ""
print("\nThen, from a terminal with push access:")
print(f'  git add results/figures/train_val_gap_{OWNER}{suffix}.json')
print(f'  git commit -m "Step 1: train/val gap, {OWNER} half{" (" + TAG + ")" if TAG else ""}"')
print("  git push")